# 規則で誤りを注入して GEC の擬似学習データを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_gec_education/synthetic_errors.ipynb)

NLP クックブック 17-07 ／ 文法誤り訂正と言語教育

文法誤り訂正の学習データが足りない。正しい文に人工的な誤りを入れて、(誤り文, 正しい文) のペアを大量に作りたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import random
from fugashi import Tagger

tagger = Tagger()
PARTICLES = ["が", "を", "に", "で", "は", "と", "へ", "の"]

def corrupt(sentence, rng, p=0.15):
    """学習者に多い誤りを模して、助詞の置換・脱落・余剰と、活用語尾の誤りを確率的に入れる"""
    out = []
    for w in tagger(sentence):
        surface, pos1, pos2 = w.surface, w.feature.pos1, w.feature.pos2
        if pos1 == "助詞" and pos2 in ("格助詞", "係助詞") and surface in PARTICLES and rng.random() < p:
            r = rng.random()
            if r < 0.6:
                out.append(rng.choice([x for x in PARTICLES if x != surface]))   # 置換
            elif r < 0.8:
                pass                                                              # 脱落
            else:
                out += [surface, rng.choice(PARTICLES)]                           # 余剰
        elif pos1 == "動詞" and w.feature.cForm and w.feature.cForm.startswith("連用形") and rng.random() < p:
            out.append(w.feature.orthBase or surface)                             # 活用させずに辞書形のままにする
        else:
            out.append(surface)
    return "".join(out)

rng = random.Random(0)
clean = ["私は昨日友達と映画を見に行きました。", "図書館で日本語の本を読みます。", "母が作った料理はとてもおいしいです。", "来週の月曜日に東京へ出張します。"]
for s in clean:
    for _ in range(2):
        noisy = corrupt(s, rng, p=0.3)
        if noisy != s:
            print(f"{noisy}  →  {s}")

私は昨日友達と映画を見るに行きました。  →  私は昨日友達と映画を見に行きました。
図書館日本語の本をに読みます。  →  図書館で日本語の本を読みます。
母と作った料理でとてもおいしいです。  →  母が作った料理はとてもおいしいです。
母が作るた料理はとてもおいしいです。  →  母が作った料理はとてもおいしいです。
来週の月曜日東京へ出張します。  →  来週の月曜日に東京へ出張します。
来週の月曜日の東京へ出張します。  →  来週の月曜日に東京へ出張します。


- 擬似データで事前学習してから実データでファインチューニングするのが、GEC の定番の手順。
- 誤りの種類と割合は、実際の学習者コーパスでの分布に合わせるほど効果が高い。ここの規則は最小限の例。
- 逆翻訳 (正しい文 → 誤り文を生成するモデル) や、LLM に誤りを書かせる方法もある。